In [1]:
import os
import shutil
from google.colab import drive
drive.mount('/content/drive')

# Define model name and paths
model_name = "phi4" # change this
drive_folder = "/content/drive/MyDrive/Models/finetunes/" + model_name + "/"
drive_gguf_path = drive_folder + "phi-4.Q8_0" + ".gguf" # and this
dataset = "pss14_valid_dataset.json" # and this
drive_dataset_path = "/content/drive/MyDrive/Models/" + dataset

local_gguf_path = "/content/" + model_name + ".gguf"
local_dataset_path = "/content/" + dataset

# Copy GGUF file to local runtime
print(f"Copying model from Google Drive to local runtime...")
if not os.path.exists(local_gguf_path):
    shutil.copy2(drive_gguf_path, local_gguf_path)

# Copy dataset to local runtime
print(f"Copying dataset from Google Drive to local runtime...")
if os.path.exists(drive_dataset_path):
    shutil.copy2(drive_dataset_path, local_dataset_path)
else:
    print("Warning: Dataset not found in Drive directory.")

print("Copy complete!")

Mounted at /content/drive
Copying model from Google Drive to local runtime...
Copying dataset from Google Drive to local runtime...
Copy complete!


In [2]:
!sudo apt update
!sudo apt install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,341 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,292 kB]
Get:13 http://arch

In [3]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [4]:
# Create the Modelfile pointing to the local file
modelfile_content = f"FROM {local_gguf_path}\n"
with open("Modelfile", "w") as f:
    f.write(modelfile_content)

# Create the model in Ollama
!ollama create {model_name} -f /content/Modelfile

# List available models to verify
!ollama list


NAME           ID              SIZE     MODIFIED               
phi4:latest    4004b5a7a173    15 GB    Less than a second ago    


In [5]:
%pip install openai scikit-learn pandas

In [6]:
# Create client

from openai import OpenAI
from sklearn.model_selection import train_test_split
import pandas as pd

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",  # Ollama
    api_key="sk-1234"  # Dummy key
)

In [7]:
SYSTEM_PROMPT = """
You are a PSS-14 survey scoring assistant.
Your job is to extract answers for the PSS-14 based on a given conversation transcript.
The Perceived Stress Scale 14-item version (PSS-14) is a clinical assessment tool that measures how often a person has felt or thought a certain way over the past month.

The conversation asks questions in this specific order. Map each user response to the correct q number:
q1   Upset because of something that happened unexpectedly
q2   Felt unable to control the important things in your life
q3   Felt nervous or stressed
q4   Dealt successfully with irritating everyday hassles            (positive item)
q5   Felt that you were effectively coping with important changes    (positive item)
q6   Felt confident about your ability to handle personal problems   (positive item)
q7   Felt that things were going your way                            (positive item)
q8   Found that you could not cope with all the things you had to do
q9   Been able to control irritations in your life                   (positive item)
q10  Felt that you were on top of things                             (positive item)
q11  Been angered by things that happened outside your control
q12  Found yourself thinking about things you still have to accomplish
q13  Been able to control the way you spend your time                (positive item)
q14  Felt that difficulties were piling up so high you could not overcome them

Scale mapping:
0 = never
1 = almost never
2 = sometimes
3 = fairly often
4 = very often

Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The final normalized answer for each question must be a single string value: "0", "1", "2", "3", or "4".
Rules:
- Read each user response in order. The first user response answers q1, the second answers q2, the third answers q3, and so on through q14.
- Score each response based solely on the frequency the user reported (do NOT reverse-score positive items here; record the literal frequency they expressed).
- Map natural-language frequencies to the closest scale value (e.g., "never" -> 0, "almost never" / "rarely" -> 1, "sometimes" / "occasionally" -> 2, "fairly often" / "often" -> 3, "very often" / "almost always" / "constantly" -> 4).
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any q1-q14 key.
Return JSON in exactly this shape:
{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "0",
  "q7": "0",
  "q8": "0",
  "q9": "0",
  "q10": "0",
  "q11": "0",
  "q12": "0",
  "q13": "0",
  "q14": "0"
}
"""
MODEL = model_name

- Use a json dataset of PSS-14 conversation transcripts
- Extract scores using LLM
- Validate whether they were correct using MSE

In [8]:
# Load the PSS-14 dataset
import json

with open('pss14_valid_dataset.json') as f:
    data = json.loads('\n'.join([line for line in f]))

# Split the data into conversation data and expected values
conversations = [entry['conversation'] for entry in data]
expected_outputs = [entry['expected_output'] for entry in data]

print('SUCCESS: Number of conversations matches number of scores' if len(conversations) == len(expected_outputs) else 'FAIL: Number of conversations does not match number of scores')
print(f'Total conversations: {len(conversations)}')

SUCCESS: Number of conversations matches number of scores
Total conversations: 10


In [9]:
# Further split the data into training and test data
conv_train, _conv_test, score_train, _score_test = train_test_split(conversations, expected_outputs, random_state=1234)
print(f'Train: {len(conv_train)}, Test: {len(_conv_test)}')

Train: 7, Test: 3


In [10]:
def Message(content, role='user'):
    return {
        'role': role,
        'content': content
    }

def create_prediction(transcript) -> str:
    completion = client.chat.completions.create(
        model=MODEL,
        temperature=0.1,  # low temperature for consistent scoring
        messages=[
            Message(SYSTEM_PROMPT, 'system'),
            Message(json.dumps(transcript))
        ],
    )
    return completion.choices[0].message.content

In [11]:
# Run predictions on training and test set
def generate_scores(conversations):
    scores = []
    for i, c in enumerate(conversations):
        print(f'Generating {i+1}/{len(conversations)}')
        scores.append(create_prediction(c))
    return scores

train_scores_raw = generate_scores(conv_train)
test_scores_raw = generate_scores(_conv_test)

Generating 1/7
Generating 2/7
Generating 3/7
Generating 4/7
Generating 5/7
Generating 6/7
Generating 7/7
Generating 1/3
Generating 2/3
Generating 3/3


In [12]:
import json, re

# Parse json scores
def parse_scores(raw_scores):
    parsed = []
    for s in raw_scores:
        # Remove markdown JSON fences if present
        cleaned_s = re.sub(r'^```json\s*', '', s)
        cleaned_s = re.sub(r'^```\s*', '', cleaned_s)
        cleaned_s = re.sub(r'\s*```$', '', cleaned_s)
        cleaned_s = cleaned_s.strip()

        try:
            parsed.append(json.loads(cleaned_s))
        except json.JSONDecodeError:
            print(f'Failed to parse: {cleaned_s}')
            parsed.append({f'q{i}': '0' for i in range(1, 15)})
    return parsed

train_scores_parsed = parse_scores(train_scores_raw)
test_scores_parsed = parse_scores(test_scores_raw)

In [13]:
from sklearn.metrics import mean_squared_error

def convert_scores_to_array(scores_df):
    return [int(s) if s else 0 for s in scores_df.values.flatten().tolist()]

def evaluate_split(parsed_scores, expected_scores, name):
    pred_df = pd.DataFrame(parsed_scores)
    expected_df = pd.DataFrame(expected_scores)

    cols = [f'q{i}' for i in range(1, 15)]
    pred_df = pred_df[cols]
    expected_df = expected_df[cols]

    mse = mean_squared_error(
        convert_scores_to_array(expected_df),
        convert_scores_to_array(pred_df)
    )
    print(f'{name} MSE: {mse}')

    display(expected_df.compare(pred_df, align_axis=1, keep_shape=True, keep_equal=True).rename(
        columns={'self': 'Expected', 'other': 'Predicted'}, level=1
    ))

evaluate_split(train_scores_parsed, score_train, 'Train')
evaluate_split(test_scores_parsed, _score_test, 'Test')

Train MSE: 0.05102040816326531


q1                 q2                 q3                 q4            \
  Expected Predicted Expected Predicted Expected Predicted Expected Predicted   
0        2         2        1         1        2         2        3         3   
1        1         1        0         0        1         1        3         3   
2        3         3        3         3        3         3        1         1   
3        3         3        4         4        4         4        0         0   
4        1         1        1         1        2         2        3         3   
5        2         2        2         2        2         2        2         2   
6        3         3        2         2        3         3        1         1   

        q5            ...      q10                q11                q12  \
  Expected Predicted  ... Expected Predicted Expected Predicted Expected   
0        3         3  ...        2         2        2         2        3   
1        4         4  ...        3         3        1         1        2   
2        1         1  ...        1         1        1         3        3   
3        0         0  ...        0         0        3         3        4   
4        3         3  ...        2         2        1         1        2   
5        2         2  ...        2         2        2         2        3   
6        1         1  ...        1         1        3         3        3   

                 q13                q14            
  Predicted Expected Predicted Expected Predicted  
0         3        2         2        1         1  
1         2        3         3        0         0  
2         4        2         2        3         3  
3         4        0         0        4         4  
4         2        2         2        0         0  
5         3        2         2        1         1  
6         3        1         1        3         3  

[7 rows x 28 columns]

Test MSE: 0.0


q1                 q2                 q3                 q4            \
  Expected Predicted Expected Predicted Expected Predicted Expected Predicted   
0        2         2        3         3        3         3        1         1   
1        2         2        2         2        3         3        2         2   
2        0         0        0         0        1         1        4         4   

        q5            ...      q10                q11                q12  \
  Expected Predicted  ... Expected Predicted Expected Predicted Expected   
0        1         1  ...        1         1        2         2        3   
1        2         2  ...        2         2        2         2        3   
2        4         4  ...        3         3        1         1        2   

                 q13                q14            
  Predicted Expected Predicted Expected Predicted  
0         3        1         1        2         2  
1         3        2         2        2         2  
2         2        3         3        0         0  

[3 rows x 28 columns]